# 03 — Modelado: King Crimson

Orden obligatorio: baseline naive estacional primero (`seasonal_naive_forecast`)
— si SARIMA/LightGBM no le ganan, no hay proyecto. Validar con
`expanding_window_splits`, nunca con un split aleatorio (fuga de futuro en
series temporales). Comparar MAE/RMSE/MAPE de los tres por isla.

In [ ]:
from src.model import evaluate_forecast, expanding_window_splits, fit_lightgbm, fit_sarima, seasonal_naive_forecast

TARGET = "revpar_eur"
HORIZON = 3

# df_features ya viene de 02_decomposition_features — en Colab, reejecutar
# esas celdas aquí o cargar desde data/processed/ si se exportó.
resultados_por_isla = {}

for isla, grupo in df_features.groupby("isla"):
    serie = grupo.set_index("fecha")[TARGET].dropna()
    splits = expanding_window_splits(len(serie), min_train_size=36, horizon=HORIZON)

    for train_end, test_end in splits:
        train, test = serie.iloc[:train_end], serie.iloc[train_end:test_end]

        naive_pred = seasonal_naive_forecast(train, horizon=HORIZON)
        naive_result = evaluate_forecast(test.values, naive_pred, "naive_estacional")

        sarima_fit = fit_sarima(train)
        sarima_pred = sarima_fit.forecast(steps=HORIZON)
        sarima_result = evaluate_forecast(test.values, sarima_pred.values, "sarima")

        resultados_por_isla.setdefault(isla, []).extend([naive_result, sarima_result])

    print(f"{isla}: {len(splits)} folds evaluados")